# Raw to Bronze load

Loads landed files into the four Bronze tables. **Normal run: Run all with the default widgets.** Backfill: set `source_dir=processed` and a date or batch range. Replay from the start: `replay=true`. Details in the guide, sections 2A and 8.

**Bronze is schema on read:** every source data column is stored as text (`STRING`); only `collection_timestamp` and `load_timestamp` are timestamps. New or missing columns are logged and the load carries on. Casting, type drift and the quarantine of values that cannot be cast belong to Silver.

### CELL 1 - Parameters and configuration

In [0]:
# CELL 1 - Parameters and configuration
import csv, hashlib, json, os, re, tarfile, time, traceback, uuid
from datetime import date, datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (BooleanType, ByteType, DateType, DoubleType, FloatType, IntegerType,
                               LongType, ShortType, StringType, StructField, StructType, TimestampType)

dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("bronze_schema", "bronze", "bronze_schema (use bronze_test for the proof notebook)")
dbutils.widgets.dropdown("source", "all", ["all", "arena", "cooperbench", "benchlm", "epoch"], "source")
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"], "load_type")
dbutils.widgets.text("source_dir", "new", "source_dir: new | processed | /Volumes/...")
dbutils.widgets.text("batch_id", "", "batch_id (one batch; empty = every batch)")
dbutils.widgets.text("from_batch", "", "from_batch (first batch id or prefix, e.g. 20261005T000000Z)")
dbutils.widgets.text("to_batch", "", "to_batch (last batch id or prefix, inclusive)")
dbutils.widgets.text("start_date", "", "start_date YYYY-MM-DD (empty = no limit)")
dbutils.widgets.text("end_date", "", "end_date YYYY-MM-DD (empty = no limit)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "force_reload")
dbutils.widgets.dropdown("replay", "false", ["false", "true"],
                         "replay: rebuild from the first batch, one batch at a time, oldest first")

P = {k: dbutils.widgets.get(k).strip() for k in
     ["catalog", "bronze_schema", "source", "load_type", "source_dir", "batch_id", "from_batch", "to_batch",
      "start_date", "end_date", "force_reload", "replay"]}

PARAM_ERRORS = []


def parse_day(name):
    value = P[name]
    if not value:
        return ""
    try:
        return date.fromisoformat(value).isoformat()
    except ValueError:
        PARAM_ERRORS.append(f"{name}='{value}' is not a valid YYYY-MM-DD date")
        return ""


START, END = parse_day("start_date"), parse_day("end_date")
if START and END and START > END:
    PARAM_ERRORS.append(f"start_date {START} is after end_date {END}")

FROM_B, TO_B = P["from_batch"], P["to_batch"]
if FROM_B and TO_B:
    n = min(len(FROM_B), len(TO_B))
    if FROM_B[:n] > TO_B[:n]:
        PARAM_ERRORS.append(f"from_batch {FROM_B} is later than to_batch {TO_B}")

REPLAY = P["replay"] == "true"
if REPLAY and P["source_dir"] == "new":
    P["source_dir"] = "processed"
FORCE = P["force_reload"] == "true" or REPLAY

CATALOG, BSCHEMA = P["catalog"], P["bronze_schema"]
for _name in (CATALOG, BSCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", _name):
        raise ValueError(f"Invalid catalog/schema name: {_name!r}")
ROOT = f"/Volumes/{CATALOG}/landing/staging/frontier"
SRC_DIR = P["source_dir"].rstrip("/") if P["source_dir"].startswith("/") else f"{ROOT}/{P['source_dir']}"
IS_LIVE = SRC_DIR == f"{ROOT}/new"

MIN_FULL_BYTES = 150_000_000
MIN_INCREMENTAL_BYTES = 1_000_000
MEMBER_EXTENSIONS = {".json", ".jsonl"}
MAX_MEMBER_BYTES = 64_000_000
BATCH_BYTES = 16_000_000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
spark.conf.set("spark.sql.session.timeZone", "UTC")

T_RUNS = f"{CATALOG}.ops.pipeline_execution_logs"
T_ERRORS = f"{CATALOG}.ops.error_logs"
T_QUAR = f"{CATALOG}.ops.quarantine_log"
T_DRIFT = f"{CATALOG}.ops.schema_drift_log"
T_ARENA = f"{CATALOG}.{BSCHEMA}.arena_leaderboard_raw"
T_COOP = f"{CATALOG}.{BSCHEMA}.cooperbench_member_raw"
T_BLM = f"{CATALOG}.{BSCHEMA}.benchlm_raw"
T_EPOCH = f"{CATALOG}.{BSCHEMA}.epoch_models_raw"

print("Run id:", RUN_ID, "| parameters:", P, "| reading from:", SRC_DIR)

Run id: 20261010T114341Z_0c52a6 | parameters: {'catalog': 'workspace', 'bronze_schema': 'bronze', 'source': 'all', 'load_type': 'all', 'source_dir': 'new', 'batch_id': '', 'from_batch': '', 'to_batch': '', 'start_date': '', 'end_date': '', 'force_reload': 'false', 'replay': 'false'} | reading from: /Volumes/workspace/landing/staging/frontier/new


### CELL 2 - Explicit schemas (the data contracts). Nothing is inferred anywhere in this notebook.

In [0]:
# CELL 2 - Explicit schemas (the data contracts). Nothing is inferred anywhere in this notebook.

ARENA_COLUMNS = [
    "model_name", "organization", "license", "rating", "variance", "rating_q975", "rating_q025",
    "vote_count", "rank", "category", "leaderboard_publish_date",
]
ARENA_CONTRACT = StructType([StructField(n, StringType(), True) for n in ARENA_COLUMNS])
ARENA_META = ["subset", "load_type", "raw_payload", "record_hash", "source_file", "batch_id",
              "collection_timestamp", "load_timestamp"]
ARENA_CORE = ("model_name", "rank", "leaderboard_publish_date")

EPOCH_CONTRACT = StructType([
    StructField(n, StringType(), True)
    for n in [
        "Model",
        "Domain",
        "Task",
        "Organization",
        "Authors",
        "Publication date",
        "Reference",
        "Link",
        "Citations",
        "Notability criteria"
    ]
])
COOP_ROW_SCHEMA = StructType([
    StructField("study_setting", StringType(), True), StructField("archive_file", StringType(), True),
    StructField("member_path", StringType(), True), StructField("member_extension", StringType(), True),
    StructField("member_size_bytes", StringType(), True), StructField("raw_content", StringType(), True),
    StructField("record_hash", StringType(), True), StructField("ingested_at", StringType(), True)])

BLM_ROW_SCHEMA = StructType([
    StructField("dataset", StringType(), True), StructField("record_path", StringType(), True),
    StructField("record_key", StringType(), True), StructField("record_index", StringType(), True),
    StructField("raw_payload", StringType(), True), StructField("payload_hash", StringType(), True),
    StructField("record_hash", StringType(), True), StructField("ingested_at", StringType(), True)])

### CELL 3 - Create the tables (every table has load_timestamp)

In [0]:
# CELL 3 - Create the tables (every table has load_timestamp)

OPS_SPECS = {
    T_RUNS: [("run_id", "string"), ("batch_id", "string"), ("layer", "string"), ("source", "string"),
             ("dataset", "string"), ("load_type", "string"), ("file_or_param", "string"),
             ("parameters_json", "string"), ("start_time", "timestamp"), ("end_time", "timestamp"),
             ("duration_seconds", "double"), ("status", "string"), ("rows_read", "bigint"),
             ("rows_inserted", "bigint"), ("rows_updated", "bigint"), ("rows_skipped_duplicate", "bigint"),
             ("rows_quarantined", "bigint"), ("bytes_processed", "bigint"), ("message", "string"),
             ("load_timestamp", "timestamp")],
    T_ERRORS: [("error_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("component", "string"),
               ("layer", "string"), ("source", "string"), ("file_or_param", "string"), ("severity", "string"),
               ("error_type", "string"), ("error_message", "string"), ("http_status", "bigint"),
               ("attempt", "bigint"), ("stack_trace", "string"), ("occurred_at", "timestamp"),
               ("resolved", "boolean"), ("load_timestamp", "timestamp")],
    T_QUAR: [("quarantine_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("source", "string"),
             ("dataset", "string"), ("source_file", "string"), ("item", "string"), ("reason", "string"),
             ("detail", "string"), ("size_bytes", "bigint"), ("record_sample", "string"),
             ("quarantined_at", "timestamp"), ("load_timestamp", "timestamp")],
    T_DRIFT: [("drift_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("dataset", "string"),
              ("source_file", "string"), ("column_name", "string"), ("event_type", "string"),
              ("expected_type", "string"), ("actual_type", "string"), ("action_taken", "string"),
              ("detected_at", "timestamp"), ("load_timestamp", "timestamp")],
}
OPS_TYPES = {"string": StringType(), "bigint": LongType(), "double": DoubleType(), "boolean": BooleanType()}

for schema in (BSCHEMA, "ops"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for table, spec in OPS_SPECS.items():
    ddl = ", ".join(f"`{n}` {t.upper()}" for n, t in spec)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({ddl}) USING DELTA")
    have = {c.lower() for c in spark.table(table).columns}
    for n, t in spec:
        if n.lower() not in have:
            spark.sql(f"ALTER TABLE {table} ADD COLUMNS (`{n}` {t.upper()})")

arena_cols = ", ".join(f"`{f.name}` {f.dataType.simpleString().upper()}" for f in ARENA_CONTRACT.fields)
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_ARENA} (
    subset STRING, load_type STRING, {arena_cols},
    raw_payload STRING, record_hash STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_COOP} (
    study_setting STRING, archive_file STRING, member_path STRING, member_extension STRING,
    member_size_bytes STRING, raw_content STRING, record_hash STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_BLM} (
    dataset STRING, record_path STRING, record_key STRING, record_index STRING, raw_payload STRING,
    payload_hash STRING, record_hash STRING, snapshot_id STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

epoch_cols = ", ".join("`" + f.name + "` STRING" for f in EPOCH_CONTRACT.fields)
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_EPOCH} (
    {epoch_cols},
    raw_payload STRING, payload_hash STRING, record_hash STRING, snapshot_id STRING, source_file STRING,
    batch_id STRING, collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA
    TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2',
                   'delta.minWriterVersion' = '5')""")

_typed_cols = [f"{f.name}:{f.dataType.simpleString()}" for f in spark.table(T_ARENA).schema.fields
               if f.name not in ARENA_META and not isinstance(f.dataType, StringType)]
if _typed_cols:
    raise RuntimeError(
        f"{T_ARENA} still has typed source columns ({', '.join(_typed_cols[:5])} ...). Bronze is now text-only. "
        f"One-time fix: run  DROP TABLE {T_ARENA}  and then replay the history from 'processed' "
        f"(replay = true, source = arena). See the guide, Step 15A.")

_epoch_have = {f.name.lower() for f in spark.table(T_EPOCH).schema.fields}
_epoch_missing = [f.name for f in EPOCH_CONTRACT.fields if f.name.lower() not in _epoch_have]
if _epoch_missing:
    raise RuntimeError(
        f"{T_EPOCH} has no text columns for: {', '.join(_epoch_missing[:5])} ... Bronze now keeps every CSV column "
        f"as text. One-time fix: run  DROP TABLE {T_EPOCH}  and then replay from 'processed' "
        f"(replay = true, source = epoch). See the guide, Step 15A.")

for _t, _col, _src in ((T_COOP, "member_size_bytes", "cooperbench"), (T_BLM, "record_index", "benchlm")):
    _type = [f.dataType for f in spark.table(_t).schema.fields if f.name == _col][0]
    if not isinstance(_type, StringType):
        raise RuntimeError(f"{_t}.{_col} is {_type.simpleString()}, Bronze is text-only. Drop {_t} and replay it "
                           f"from 'processed' (replay = true, source = {_src}). See the guide, Step 15A.")

print("Tables ready.")

Tables ready.


### CELL 4 - Logging helpers (execution log, error log, quarantine log, drift log)

In [0]:
# CELL 4 - Logging helpers (execution log, error log, quarantine log, drift log)

def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def append_record(table, record):
    spec = OPS_SPECS[table]
    schema = StructType([StructField(n, StringType() if t == "timestamp" else OPS_TYPES[t], True) for n, t in spec])
    df = spark.createDataFrame([tuple(record.get(n) for n, _ in spec)], schema)
    for name, kind in spec:
        if kind == "timestamp":
            df = df.withColumn(name, F.to_timestamp(F.col(name)))
    df.write.mode("append").saveAsTable(table)


def log_error(stage, source, file_or_param, message, severity="ERROR", error_type="", batch_id="",
              stack="", http_status=None, attempt=None, component="bronze_notebook", occurred_at=None):
    print(f"{severity} | {stage} | {source} | {file_or_param} | {message}")
    try:
        append_record(T_ERRORS, {
            "error_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id, "component": component,
            "layer": stage, "source": source, "file_or_param": file_or_param, "severity": severity,
            "error_type": error_type, "error_message": str(message)[:4000], "http_status": http_status,
            "attempt": attempt, "stack_trace": stack[-4000:], "occurred_at": occurred_at or now_iso(),
            "resolved": False, "load_timestamp": now_iso()})
    except Exception as exc:
        print("Could not write to the error table:", exc)


def log_quarantine(batch_id, source, dataset, source_file, item, reason, detail="", size_bytes=None, sample=""):
    print(f"QUARANTINE | {source} | {item} | {reason}")
    append_record(T_QUAR, {"quarantine_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id,
                           "source": source, "dataset": dataset, "source_file": source_file, "item": item,
                           "reason": reason, "detail": str(detail)[:2000], "size_bytes": size_bytes,
                           "record_sample": str(sample)[:4000],
                           "quarantined_at": now_iso(), "load_timestamp": now_iso()})


def log_drift(batch_id, dataset, source_file, column, event, expected, actual, action):
    append_record(T_DRIFT, {"drift_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id,
                            "dataset": dataset, "source_file": source_file, "column_name": column,
                            "event_type": event, "expected_type": expected, "actual_type": actual,
                            "action_taken": action, "detected_at": now_iso(), "load_timestamp": now_iso()})


class QuarantineFile(Exception):
    def __init__(self, message, sample=""):
        super().__init__(message)
        self.sample = sample


RUN_TOTALS = {"rows_read": 0, "rows_inserted": 0, "rows_updated": 0, "rows_skipped": 0,
              "rows_quarantined": 0, "bytes": 0, "outcomes": {}}


class Step:
    def __init__(self, layer, source, dataset, load_type, file_or_param, batch_id, extra=None):
        self.layer, self.source, self.dataset, self.load_type = layer, source, dataset, load_type
        self.file_or_param, self.batch_id = file_or_param, batch_id
        self.params = dict(P, **(extra or {}))
        self.start = now_iso()
        self.t0 = time.time()
        self.rows_read = self.rows_inserted = self.rows_updated = 0
        self.rows_skipped = self.rows_quarantined = self.bytes = 0

    def finish(self, status, message=""):
        append_record(T_RUNS, {
            "run_id": RUN_ID, "batch_id": self.batch_id, "layer": self.layer, "source": self.source,
            "dataset": self.dataset, "load_type": self.load_type, "file_or_param": self.file_or_param,
            "parameters_json": json.dumps(self.params), "start_time": self.start, "end_time": now_iso(),
            "duration_seconds": round(time.time() - self.t0, 2), "status": status,
            "rows_read": self.rows_read, "rows_inserted": self.rows_inserted, "rows_updated": self.rows_updated,
            "rows_skipped_duplicate": self.rows_skipped, "rows_quarantined": self.rows_quarantined,
            "bytes_processed": self.bytes, "message": str(message)[:2000], "load_timestamp": now_iso()})
        if self.layer == "Raw-to-Bronze":
            for key, value in (("rows_read", self.rows_read), ("rows_inserted", self.rows_inserted),
                               ("rows_updated", self.rows_updated), ("rows_skipped", self.rows_skipped),
                               ("rows_quarantined", self.rows_quarantined), ("bytes", self.bytes)):
                RUN_TOTALS[key] += value
        print(f"{status} | {self.layer} | {self.file_or_param} | read {self.rows_read} | inserted "
              f"{self.rows_inserted} | updated {self.rows_updated} | skipped {self.rows_skipped}")

### CELL 5 - Helper functions

In [0]:
# CELL 5 - Helper functions

from pyspark.sql.types import ArrayType, BinaryType, DecimalType, MapType


def local(path):
    return path.replace("dbfs:", "", 1)


def exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def sha256_file(path):
    digest = hashlib.sha256()
    with open(local(path), "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sql_id(name):
    return "`" + name.replace("`", "``") + "`"


def move(src, dst):
    if exists(dst):
        dbutils.fs.rm(dst)
    dbutils.fs.mv(src, dst)


def merge_insert_only(df, target, key="record_hash"):
    view = "src_" + uuid.uuid4().hex[:10]
    df.createOrReplaceTempView(view)
    cols = df.columns
    insert_cols = ", ".join(sql_id(c) for c in cols)
    insert_vals = ", ".join("s." + sql_id(c) for c in cols)
    result = spark.sql(f"""
        MERGE INTO {target} AS t USING {view} AS s ON t.{key} = s.{key}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})""").collect()[0].asDict()
    spark.catalog.dropTempView(view)
    return int(result.get("num_inserted_rows") or 0), int(result.get("num_updated_rows") or 0)


def batch_columns(df, meta):
    out = (df.withColumn("source_file", F.lit(meta["source_file"]))
              .withColumn("batch_id", F.lit(meta["batch_id"]))
              .withColumn("collection_timestamp", F.to_timestamp(F.lit(meta["created_at"]))))
    if "load_timestamp" not in df.columns:
        out = out.withColumn("load_timestamp", F.current_timestamp())
    return out


def arrow_to_spark(arrow_type):
    import pyarrow as pa
    t = pa.types
    if t.is_dictionary(arrow_type):
        return arrow_to_spark(arrow_type.value_type)
    simple = [(t.is_string, StringType()), (t.is_large_string, StringType()), (t.is_int8, ByteType()),
              (t.is_int16, ShortType()), (t.is_int32, IntegerType()), (t.is_int64, LongType()),
              (t.is_uint8, ShortType()), (t.is_uint16, IntegerType()), (t.is_uint32, LongType()),
              (t.is_uint64, DecimalType(20, 0)), (t.is_float32, FloatType()), (t.is_float64, DoubleType()),
              (t.is_boolean, BooleanType()), (t.is_date32, DateType()), (t.is_binary, BinaryType()),
              (t.is_large_binary, BinaryType())]
    for check, spark_type in simple:
        if check(arrow_type):
            return spark_type
    if t.is_timestamp(arrow_type):
        return TimestampType()
    if t.is_decimal(arrow_type):
        return DecimalType(arrow_type.precision, arrow_type.scale)
    if t.is_list(arrow_type) or t.is_large_list(arrow_type):
        return ArrayType(arrow_to_spark(arrow_type.value_type), True)
    if t.is_map(arrow_type):
        return MapType(arrow_to_spark(arrow_type.key_type), arrow_to_spark(arrow_type.item_type), True)
    if t.is_struct(arrow_type):
        return StructType([StructField(arrow_type.field(i).name, arrow_to_spark(arrow_type.field(i).type), True)
                           for i in range(arrow_type.num_fields)])
    raise ValueError(f"unsupported Parquet type: {arrow_type}")


def footer_schema(path):
    import pyarrow.parquet as pq
    return StructType([StructField(f.name, arrow_to_spark(f.type), True) for f in pq.read_schema(local(path))])


def is_complex(data_type):
    return isinstance(data_type, (StructType, ArrayType, MapType))


def arena_prepare(path, meta):
    batch, dataset, rel = meta["batch_id"], f"arena/{meta['dataset']}", meta["source_file"]
    table_cols = {f.name.lower(): f.name for f in spark.table(T_ARENA).schema.fields if f.name not in ARENA_META}
    try:
        footer = footer_schema(path)
    except ValueError as exc:
        raise QuarantineFile(str(exc))
    actual = {f.name.lower(): f for f in footer.fields}

    core_missing = [c for c in ARENA_CORE if c not in actual]
    if core_missing:
        raise QuarantineFile("core columns missing: " + ", ".join(core_missing),
                             sample="file columns: " + ", ".join(f"{f.name}:{f.dataType.simpleString()}"
                                                                 for f in footer.fields))

    for low, name in table_cols.items():
        if low not in actual:
            log_drift(batch, dataset, rel, name, "MISSING_COLUMN", "string", "", "FILLED_WITH_NULL")
    new_cols = [f for low, f in actual.items() if low not in table_cols]
    for f in new_cols:
        spark.sql(f"ALTER TABLE {T_ARENA} ADD COLUMNS ({sql_id(f.name)} STRING)")
        log_drift(batch, dataset, rel, f.name, "NEW_COLUMN", "", f.dataType.simpleString(), "TABLE_EVOLVED")
    for f in footer.fields:
        if is_complex(f.dataType):
            log_drift(batch, dataset, rel, f.name, "COMPLEX_TYPE", "string", f.dataType.simpleString(),
                      "STORED_AS_JSON")

    def as_text(field):
        column = F.col(sql_id(field.name))
        return F.to_json(column) if is_complex(field.dataType) else column.cast("string")

    outputs = []
    for low, name in table_cols.items():
        outputs.append(as_text(actual[low]).alias(name) if low in actual else F.lit(None).cast("string").alias(name))
    for f in new_cols:
        outputs.append(as_text(f).alias(f.name))
    return footer, outputs

### CELL 6 - Pick up the collector's logs (GitHub -> Databricks) so every error is in one table

In [0]:
# CELL 6 - Pick up the collector's logs (GitHub -> Databricks) so every error is in one table

def ingest_collector_logs():
    pending, done = f"{ROOT}/logs/pending", f"{ROOT}/logs/ingested"
    if not exists(pending):
        return
    for f in dbutils.fs.ls(pending):
        try:
            with open(local(f.path), encoding="utf-8") as handle:
                if f.name.startswith("collector_errors_") and f.name.endswith(".jsonl"):
                    for line in handle:
                        if not line.strip():
                            continue
                        e = json.loads(line)
                        append_record(T_ERRORS, {
                            "error_id": e["error_id"], "run_id": RUN_ID, "batch_id": e.get("batch_id"),
                            "component": "collector", "layer": e.get("stage"), "source": e.get("source"),
                            "file_or_param": e.get("file_or_param"), "severity": e.get("severity"),
                            "error_type": e.get("error_type"), "error_message": e.get("message"),
                            "http_status": e.get("http_status"), "attempt": e.get("attempt"),
                            "stack_trace": e.get("traceback"), "occurred_at": e.get("occurred_at"),
                            "resolved": False, "load_timestamp": now_iso()})
                elif f.name.startswith("collector_run_") and f.name.endswith(".json"):
                    s = json.load(handle)
                    append_record(T_RUNS, {
                        "run_id": s["run_id"], "batch_id": s["batch_id"], "layer": "Source-to-Landing",
                        "source": "all", "dataset": "all", "load_type": s["mode"],
                        "file_or_param": f"github-actions:{s['mode']}", "parameters_json": json.dumps(s),
                        "start_time": s["started_at"], "end_time": s["finished_at"],
                        "duration_seconds": None, "status": s["status"], "rows_read": 0, "rows_inserted": 0,
                        "rows_updated": 0, "rows_skipped_duplicate": 0, "rows_quarantined": 0,
                        "bytes_processed": s.get("downloaded_bytes", 0), "message": s.get("message", ""),
                        "load_timestamp": now_iso()})
                else:
                    continue
            move(f.path, f"{done}/{f.name}")
            print("Collector log ingested:", f.name)
        except Exception as exc:
            log_error("ingest_collector_logs", "collector", f.name, exc, error_type=type(exc).__name__,
                      stack=traceback.format_exc())

### CELL 7 - Loader: Arena Parquet -> bronze.arena_leaderboard_raw

In [0]:
# CELL 7 - Loader: Arena Parquet -> bronze.arena_leaderboard_raw  (every source column stored as text)

def load_arena(entry, meta, path, step):
    footer, outputs = arena_prepare(path, meta)
    raw = spark.read.schema(footer).parquet(path)
    step.rows_read = raw.count()
    text = raw.select(*outputs)
    fields = sorted(text.columns, key=str.lower)
    df = (text.withColumn("raw_payload", F.to_json(F.struct(*[F.col(sql_id(c)) for c in fields])))
              .withColumn("subset", F.lit(meta["dataset"]))
              .withColumn("load_type", F.lit(meta["load_type"])))
    df = df.withColumn("record_hash", F.sha2(F.concat_ws("|", F.col("subset"), F.col("raw_payload")), 256))
    df = df.dropDuplicates(["record_hash"])
    df = df.select("subset", "load_type", *[F.col(sql_id(c)) for c in text.columns], "raw_payload", "record_hash")
    df = batch_columns(df, meta)
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_ARENA)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 8 - Loader: CooperBench archives -> bronze.cooperbench_member_raw

In [0]:
# CELL 8 - Loader: CooperBench archives -> bronze.cooperbench_member_raw

def load_cooperbench(entry, meta, path, step):
    setting, archive = meta["dataset"], os.path.basename(meta["source_file"])
    rows, chunk_bytes, ignored = [], 0, 0

    def flush():
        nonlocal rows, chunk_bytes
        if not rows:
            return
        df = spark.createDataFrame(rows, COOP_ROW_SCHEMA).drop("ingested_at")
        df = batch_columns(df.dropDuplicates(["record_hash"]), meta)
        df = df.select("study_setting", "archive_file", "member_path", "member_extension", "member_size_bytes",
                       "raw_content", "record_hash", "source_file", "batch_id", "collection_timestamp",
                       "load_timestamp")
        inserted, updated = merge_insert_only(df, T_COOP)
        step.rows_inserted += inserted
        step.rows_updated += updated
        rows, chunk_bytes = [], 0

    with tarfile.open(local(path), "r|gz") as tar:
        for member in tar:
            if not member.isfile():
                continue
            ext = os.path.splitext(member.name)[1].lower()
            if ext not in MEMBER_EXTENSIONS:
                ignored += 1
                continue
            if member.size > MAX_MEMBER_BYTES:
                log_quarantine(meta["batch_id"], "cooperbench", setting, meta["source_file"], member.name,
                               "MEMBER_TOO_LARGE", f"{member.size} bytes", member.size,
                               sample="content not read (larger than MAX_MEMBER_BYTES)")
                step.rows_quarantined += 1
                continue
            if rows and chunk_bytes + member.size > BATCH_BYTES:
                flush()
            data = tar.extractfile(member).read()
            try:
                text = data.decode("utf-8")
            except UnicodeDecodeError as exc:
                log_quarantine(meta["batch_id"], "cooperbench", setting, meta["source_file"], member.name,
                               "NOT_UTF8", exc, member.size, sample="first 200 bytes (hex): " + data[:200].hex())
                step.rows_quarantined += 1
                continue
            content_hash = hashlib.sha256(data).hexdigest()
            record_hash = hashlib.sha256(f"{setting}|{member.name}|{content_hash}".encode()).hexdigest()
            rows.append((setting, archive, member.name, ext, str(member.size), text, record_hash, now_iso()))
            step.rows_read += 1
            step.bytes += member.size
            chunk_bytes += member.size
            if chunk_bytes >= BATCH_BYTES:
                flush()
    flush()
    step.rows_skipped = step.rows_read - step.rows_inserted
    step.params["non_json_members_ignored"] = ignored

### CELL 9 - Loader: BenchLM JSON -> bronze.benchlm_raw

In [0]:
# CELL 9 - Loader: BenchLM JSON -> bronze.benchlm_raw

def split_records(doc):
    if isinstance(doc, list):
        return "$", [(None, r) for r in doc]
    if isinstance(doc, dict):
        lists = [(k, v) for k, v in doc.items() if isinstance(v, list) and v and all(isinstance(x, dict) for x in v)]
        if lists:
            key, items = max(lists, key=lambda kv: len(kv[1]))
            return "$." + key, [(None, r) for r in items]
        if len(doc) > 1 and all(isinstance(v, dict) for v in doc.values()):
            return "$", [(k, v) for k, v in doc.items()]
    return "$", [(None, doc)]


def track_columns(batch, dataset, source_file, columns):
    known = {r[0] for r in spark.sql(
        f"SELECT column_name FROM {T_DRIFT} WHERE dataset = '{dataset}' "
        f"AND event_type IN ('BASELINE','NEW_COLUMN')").collect()}
    for name in sorted(set(columns) - known):
        log_drift(batch, dataset, source_file, name, "BASELINE" if not known else "NEW_COLUMN", "", "json/string",
                  "ABSORBED_IN_RAW_PAYLOAD")


def load_benchlm(entry, meta, path, step):
    with open(local(path), encoding="utf-8") as handle:
        doc = json.load(handle)
    record_path, records = split_records(doc)
    snapshot = entry["sha256"]
    rows, columns = [], set()
    for index, (key, record) in enumerate(records):
        payload = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        if isinstance(record, dict):
            columns.update(record.keys())
        rows.append((meta["dataset"], record_path, key, str(index), payload,
                     hashlib.sha256(payload.encode()).hexdigest(),
                     hashlib.sha256(f"{snapshot}|{meta['dataset']}|{index}".encode()).hexdigest(), now_iso()))
    step.rows_read = len(rows)
    track_columns(meta["batch_id"], f"benchlm/{meta['dataset']}", meta["source_file"], columns)
    df = spark.createDataFrame(rows, BLM_ROW_SCHEMA)
    df = batch_columns(df.drop("ingested_at"), meta).withColumn("snapshot_id", F.lit(snapshot))
    df = df.select("dataset", "record_path", "record_key", "record_index", "raw_payload", "payload_hash",
                   "record_hash", "snapshot_id", "source_file", "batch_id", "collection_timestamp", "load_timestamp")
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_BLM)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 10 - Loader: Epoch CSV -> bronze.epoch_models_raw

In [0]:
# CELL 10 - Loader: Epoch CSV -> bronze.epoch_models_raw  (every CSV column stored as text, names as published)

def load_epoch(entry, meta, path, step):
    with open(local(path), encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        python_rows = sum(1 for _ in reader)
    if len(set(header)) != len(header) or len({h.lower() for h in header}) != len(header):
        raise QuarantineFile("The CSV header has duplicate column names")
    for f in EPOCH_CONTRACT.fields:
        if f.name.lower() not in {h.lower() for h in header}:
            log_drift(meta["batch_id"], "epoch/all_ai_models", meta["source_file"], f.name, "MISSING_COLUMN",
                      "string", "", "FILLED_WITH_NULL")
    schema = StructType([StructField(name, StringType(), True) for name in header])
    raw = (spark.read.schema(schema).option("header", "true").option("inferSchema", "false")
           .option("multiLine", "true").option("quote", '"').option("escape", '"').csv(path))
    step.rows_read = raw.count()
    if step.rows_read != python_rows:
        log_error("Raw-to-Bronze", "epoch", meta["source_file"],
                  f"Spark read {step.rows_read} rows but a second parser counted {python_rows}",
                  severity="WARNING", error_type="RowCountMismatch", batch_id=meta["batch_id"])
    have = {f.name.lower() for f in spark.table(T_EPOCH).schema.fields}
    for name in header:
        if name.lower() not in have:
            spark.sql(f"ALTER TABLE {T_EPOCH} ADD COLUMNS ({sql_id(name)} STRING)")
            log_drift(meta["batch_id"], "epoch/all_ai_models", meta["source_file"], name, "NEW_COLUMN", "",
                      "string", "TABLE_EVOLVED")
    snapshot = entry["sha256"]
    fields = sorted(header, key=str.lower)
    df = raw.withColumn("raw_payload", F.to_json(F.struct(*[F.col(sql_id(c)) for c in fields])))
    df = (df.withColumn("payload_hash", F.sha2("raw_payload", 256))
            .withColumn("snapshot_id", F.lit(snapshot))
            .withColumn("record_hash", F.sha2(F.concat_ws("|", F.lit(snapshot), F.col("raw_payload")), 256)))
    df = batch_columns(df.dropDuplicates(["record_hash"]), meta)
    df = df.select(*[F.col(sql_id(c)) for c in header], "raw_payload", "payload_hash", "record_hash", "snapshot_id",
                   "source_file", "batch_id", "collection_timestamp", "load_timestamp")
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_EPOCH)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 11 - The batch runner (fully parameter driven, one failure never stops the others)

In [0]:
# CELL 11 - The batch runner (fully parameter driven, one failure never stops the others)

LOADERS = {"arena": load_arena, "cooperbench": load_cooperbench, "benchlm": load_benchlm, "epoch": load_epoch}


def find_manifests():
    folder = f"{SRC_DIR}/_manifests"
    found = []
    if not exists(folder):
        return found
    for f in dbutils.fs.ls(folder):
        if not (f.name.startswith("manifest__") and f.name.endswith(".json")):
            continue
        with open(local(f.path), encoding="utf-8") as handle:
            m = json.load(handle)
        m["_path"] = local(f.path)
        day = m["created_at"][:10]
        if P["load_type"] != "all" and m["load_type"] != P["load_type"]:
            continue
        if P["batch_id"] and m["batch_id"] != P["batch_id"]:
            continue
        bid = m["batch_id"]
        if FROM_B and bid[:len(FROM_B)] < FROM_B:
            continue
        if TO_B and bid[:len(TO_B)] > TO_B:
            continue
        if (START and day < START) or (END and day > END):
            continue
        found.append(m)
    return sorted(found, key=lambda m: (m["created_at"], m["batch_id"]))


def volume_rule_ok(m):
    needed = MIN_FULL_BYTES if m["load_type"] == "full" else MIN_INCREMENTAL_BYTES
    return bool(m.get("gate_passed")) and int(m.get("landed_bytes", 0)) >= needed, needed


def already_loaded(rel, sha):
    q = spark.sql(f"SELECT 1 FROM {T_RUNS} WHERE layer = 'Raw-to-Bronze' AND status = 'SUCCESS' "
                  f"AND file_or_param = '{rel}' AND get_json_object(parameters_json, '$.sha256') = '{sha}' LIMIT 1")
    return q.count() > 0


def process_entry(m, e):
    rel = e["path"]
    src, processed, quarantined = f"{SRC_DIR}/{rel}", f"{ROOT}/processed/{rel}", f"{ROOT}/quarantine/{rel}"
    if not exists(src):
        if IS_LIVE and (exists(processed) or exists(quarantined)):
            return "DONE"
        log_error("Raw-to-Bronze", e["source"], rel, "File listed in the manifest was not found",
                  error_type="FileMissing", batch_id=m["batch_id"])
        return "FAILED"

    meta = {"batch_id": m["batch_id"], "load_type": m["load_type"], "created_at": m["created_at"],
            "dataset": e["dataset"], "source_file": rel}
    step = Step("Raw-to-Bronze", e["source"], e["dataset"], m["load_type"], rel, m["batch_id"],
                extra={"sha256": e["sha256"], "size_bytes": e["size_bytes"]})
    try:
        if os.path.getsize(local(src)) != e["size_bytes"] or sha256_file(src) != e["sha256"]:
            raise IOError("File size or SHA-256 does not match the manifest (damaged or incomplete upload)")
        step.bytes = e["size_bytes"]
        if not FORCE and already_loaded(rel, e["sha256"]):
            step.finish("SKIPPED", "Same file (same SHA-256) was already loaded successfully")
            if IS_LIVE:
                move(src, processed)
            return "SKIPPED"
        LOADERS[e["source"]](e, meta, src, step)
        step.finish("SUCCESS")
        if IS_LIVE:
            move(src, processed)
        return "LOADED"
    except QuarantineFile as q:
        step.rows_quarantined = max(step.rows_quarantined, 1)
        log_quarantine(m["batch_id"], e["source"], e["dataset"], rel, rel, "SCHEMA_CONTRACT", q, e["size_bytes"],
                       sample=q.sample)
        step.finish("QUARANTINED", q)
        if IS_LIVE:
            move(src, quarantined)
        return "QUARANTINED"
    except Exception as exc:
        log_error("Raw-to-Bronze", e["source"], rel, exc, error_type=type(exc).__name__,
                  batch_id=m["batch_id"], stack=traceback.format_exc())
        step.finish("FAILURE", exc)
        return "FAILED"


def run_pipeline():
    ingest_collector_logs()
    manifests = find_manifests()
    print("Batches to process:", len(manifests), "(oldest first)" + (" - REPLAY" if REPLAY else ""))
    for m in manifests:
        print("   ", m["created_at"], m["batch_id"], m["load_type"])
    totals = {"LOADED": 0, "SKIPPED": 0, "QUARANTINED": 0, "FAILED": 0, "DONE": 0, "REJECTED": 0}
    for m in manifests:
        batch_step = Step("Raw-to-Bronze (batch)", "all", m["batch_id"], m["load_type"], m["_path"], m["batch_id"])
        ok, needed = volume_rule_ok(m)
        if not ok:
            log_error("Raw-to-Bronze", "all", m["_path"],
                      f"Batch rejected: landed {m.get('landed_bytes')} bytes, rule needs {needed}",
                      error_type="VolumeRuleNotMet", batch_id=m["batch_id"])
            batch_step.finish("REJECTED", "Volume rule not met")
            totals["REJECTED"] += 1
            if IS_LIVE:
                move(m["_path"], f"{ROOT}/quarantine/{os.path.basename(m['_path'])}")
            continue
        outcomes = []
        for e in m["files"]:
            if P["source"] != "all" and e["source"] != P["source"]:
                continue
            outcome = process_entry(m, e)
            outcomes.append(outcome)
            totals[outcome] += 1
        batch_step.bytes = int(m.get("landed_bytes", 0))
        batch_ok = "FAILED" not in outcomes
        batch_step.finish("SUCCESS" if batch_ok else "FAILURE", f"files: {len(outcomes)}")
        if IS_LIVE and batch_ok and P["source"] == "all":
            move(m["_path"], f"{ROOT}/processed/_manifests/{os.path.basename(m['_path'])}")
    RUN_TOTALS["outcomes"] = totals
    print("Finished:", totals)
    if totals["FAILED"]:
        raise RuntimeError(f"{totals['FAILED']} file(s) failed. See {T_ERRORS}. Re-run to retry only those.")

KEY_TABLES = {"arena": T_ARENA, "cooperbench": T_COOP, "benchlm": T_BLM, "epoch": T_EPOCH}


def check_primary_keys():
    problems = []
    for source, table in KEY_TABLES.items():
        step = Step("Bronze data quality (primary key)", source, table.split(".")[-1], "check",
                    "record_hash unique and not null", "")
        df = spark.table(table)
        total = df.count()
        duplicates = df.groupBy("record_hash").count().filter("count > 1").count()
        nulls = df.filter(F.col("record_hash").isNull()).count()
        step.rows_read = total
        ok = duplicates == 0 and nulls == 0
        step.finish("SUCCESS" if ok else "FAILURE", f"rows: {total}, duplicate keys: {duplicates}, null keys: {nulls}")
        if not ok:
            problems.append(f"{table}: {duplicates} duplicate and {nulls} null record_hash")
    if problems:
        raise RuntimeError("Primary key problem: " + "; ".join(problems))


def run_with_log():
    run_step = Step("Raw-to-Bronze (run)", P["source"], "all", P["load_type"],
                    f"source_dir={SRC_DIR}" + (" | REPLAY" if REPLAY else ""),
                    P["batch_id"] or (f"{FROM_B or 'start'}..{TO_B or 'end'}" if (FROM_B or TO_B) else "all"))
    try:
        if PARAM_ERRORS:
            raise ValueError("Invalid parameter: " + "; ".join(PARAM_ERRORS))
        run_pipeline()
        if RUN_TOTALS["rows_inserted"] > 0 or REPLAY or FORCE:
            check_primary_keys()
        outcomes = RUN_TOTALS["outcomes"]
        note = "no batch matched the parameters, nothing to load" if not any(outcomes.values()) else f"files: {outcomes}"
        status = "SUCCESS"
    except Exception as exc:
        log_error("Raw-to-Bronze", P["source"], run_step.file_or_param, exc, error_type=type(exc).__name__,
                  batch_id=run_step.batch_id, stack=traceback.format_exc())
        status, note = "FAILURE", exc
        failure = exc
    else:
        failure = None
    run_step.rows_read, run_step.rows_inserted = RUN_TOTALS["rows_read"], RUN_TOTALS["rows_inserted"]
    run_step.rows_updated, run_step.rows_skipped = RUN_TOTALS["rows_updated"], RUN_TOTALS["rows_skipped"]
    run_step.rows_quarantined, run_step.bytes = RUN_TOTALS["rows_quarantined"], RUN_TOTALS["bytes"]
    run_step.finish(status, note)
    if failure is not None:
        raise failure


run_with_log()

Batches to process: 2 (oldest first)
    2026-10-10T08:14:16Z 20261010T081407Z_full_release full
    2026-10-10T08:14:16Z 20261010T081407Z_full_release_text full
SUCCESS | Raw-to-Bronze | arena/agent__full__20261010T081407Z.parquet | read 1480 | inserted 1480 | updated 0 | skipped 0
SUCCESS | Raw-to-Bronze (batch) | /Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release.json | read 0 | inserted 0 | updated 0 | skipped 0
SUCCESS | Raw-to-Bronze (batch) | /Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release_text.json | read 0 | inserted 0 | updated 0 | skipped 0
Finished: {'LOADED': 1, 'SKIPPED': 0, 'QUARANTINED': 0, 'FAILED': 0, 'DONE': 1, 'REJECTED': 0}
SUCCESS | Bronze data quality (primary key) | record_hash unique and not null | read 2200800 | inserted 0 | updated 0 | skipped 0
SUCCESS | Bronze data quality (primary key) | record_hash unique and not null | read 17328 | inserted 0 | updated 0 | 

### CELL 12 - Health check

In [0]:
# CELL 12 - Health check

def _count(folder):
    return len(dbutils.fs.ls(folder)) if exists(folder) else 0


display(spark.createDataFrame(
    [("new (waiting to be loaded)", _count(f"{ROOT}/new/_manifests")),
     ("processed (already loaded, replayable)", _count(f"{ROOT}/processed/_manifests")),
     ("quarantine (set aside)", _count(f"{ROOT}/quarantine"))],
    "staging_area string, batches_or_items long"))
display(spark.sql(f"""
    SELECT 'arena_leaderboard_raw' AS table_name, COUNT(*) AS rows_loaded, MAX(load_timestamp) AS last_load FROM {T_ARENA}
    UNION ALL SELECT 'cooperbench_member_raw', COUNT(*), MAX(load_timestamp) FROM {T_COOP}
    UNION ALL SELECT 'benchlm_raw', COUNT(*), MAX(load_timestamp) FROM {T_BLM}
    UNION ALL SELECT 'epoch_models_raw', COUNT(*), MAX(load_timestamp) FROM {T_EPOCH}"""))
display(spark.sql(f"""SELECT layer, file_or_param, status, rows_read, rows_inserted, rows_updated,
    rows_skipped_duplicate, start_time, end_time FROM {T_RUNS} ORDER BY load_timestamp DESC LIMIT 20"""))
display(spark.sql(f"""SELECT quarantined_at, source, source_file, reason, detail, record_sample FROM {T_QUAR}
    ORDER BY quarantined_at DESC LIMIT 20"""))
display(spark.sql(f"""SELECT detected_at, dataset, column_name, event_type, expected_type, actual_type, action_taken
    FROM {T_DRIFT} ORDER BY detected_at DESC LIMIT 20"""))
display(spark.sql(f"""SELECT occurred_at, component, severity, source, file_or_param, error_type, error_message
    FROM {T_ERRORS} WHERE resolved = false ORDER BY occurred_at DESC LIMIT 20"""))

staging_area,batches_or_items
new (waiting to be loaded),0
"processed (already loaded, replayable)",3
quarantine (set aside),1


table_name,rows_loaded,last_load
arena_leaderboard_raw,2200800,2026-10-10T11:44:37.254Z
cooperbench_member_raw,17328,2026-10-10T08:31:45.712Z
benchlm_raw,2383,2026-10-10T08:34:26.985Z
epoch_models_raw,3626,2026-10-10T08:36:27.211Z


layer,file_or_param,status,rows_read,rows_inserted,rows_updated,rows_skipped_duplicate,start_time,end_time
Raw-to-Bronze (run),source_dir=/Volumes/workspace/landing/staging/frontier/new,SUCCESS,1480,1480,0,0,2026-10-10T11:43:50.247Z,2026-10-10T11:45:09.371Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,3626,0,0,0,2026-10-10T11:45:06.092Z,2026-10-10T11:45:07.783Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,2383,0,0,0,2026-10-10T11:45:02.806Z,2026-10-10T11:45:04.526Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,17328,0,0,0,2026-10-10T11:44:58.682Z,2026-10-10T11:45:00.551Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,2200800,0,0,0,2026-10-10T11:44:54.975Z,2026-10-10T11:44:57.002Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release_text.json,SUCCESS,0,0,0,0,2026-10-10T11:44:51.119Z,2026-10-10T11:44:51.729Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release.json,SUCCESS,0,0,0,0,2026-10-10T11:43:52.178Z,2026-10-10T11:44:47.671Z
Raw-to-Bronze,arena/agent__full__20261010T081407Z.parquet,SUCCESS,1480,1480,0,0,2026-10-10T11:43:52.531Z,2026-10-10T11:44:44.756Z
Raw-to-Bronze (run),source_dir=/Volumes/workspace/landing/staging/frontier/new,SUCCESS,0,0,0,0,2026-10-10T11:33:54.488Z,2026-10-10T11:34:20.051Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release_text.json,SUCCESS,0,0,0,0,2026-10-10T11:34:04.819Z,2026-10-10T11:34:06.163Z


quarantined_at,source,source_file,reason,detail,record_sample
2026-10-10T08:30:53.968Z,arena,arena/agent__full__20261010T081407Z.parquet,SCHEMA_CONTRACT,7 of 13 expected columns are missing,"file columns: model_name:string, organization:string, license:string, score:double, score_ci_lower:double, score_ci_upper:double, observation_count:double, session_count:double, rank:bigint, category:string, leaderboard_publish_date:string"
2026-10-10T08:30:13.048Z,arena,arena/text_style_control__full__20261010T081407Z.parquet,SCHEMA_CONTRACT,"vote_count: table has bigint, file has double","file columns: model_name:string, organization:string, license:string, rating:double, rating_lower:double, rating_upper:double, variance:double, vote_count:double, rank:bigint, category:string, leaderboard_publish_date:string"
2026-10-10T08:29:59.801Z,arena,arena/text__full__20261010T081407Z.parquet,SCHEMA_CONTRACT,"vote_count: table has bigint, file has double; rank: table has bigint, file has double","file columns: model_name:string, organization:string, license:string, rating:double, rating_lower:double, rating_upper:double, variance:double, vote_count:double, rank:double, category:string, leaderboard_publish_date:string"


detected_at,dataset,column_name,event_type,expected_type,actual_type,action_taken
2026-10-10T11:44:31.200Z,arena/agent,session_count,NEW_COLUMN,,double,TABLE_EVOLVED
2026-10-10T11:44:27.570Z,arena/agent,observation_count,NEW_COLUMN,,double,TABLE_EVOLVED
2026-10-10T11:44:23.827Z,arena/agent,score_ci_upper,NEW_COLUMN,,double,TABLE_EVOLVED
2026-10-10T11:44:20.113Z,arena/agent,score_ci_lower,NEW_COLUMN,,double,TABLE_EVOLVED
2026-10-10T11:44:16.387Z,arena/agent,score,NEW_COLUMN,,double,TABLE_EVOLVED
2026-10-10T11:44:10.769Z,arena/agent,rating_upper,MISSING_COLUMN,double,,FILLED_WITH_NULL
2026-10-10T11:44:07.731Z,arena/agent,rating_lower,MISSING_COLUMN,double,,FILLED_WITH_NULL
2026-10-10T11:44:05.252Z,arena/agent,vote_count,MISSING_COLUMN,bigint,,FILLED_WITH_NULL
2026-10-10T11:44:02.103Z,arena/agent,rating_q025,MISSING_COLUMN,double,,FILLED_WITH_NULL
2026-10-10T11:43:59.279Z,arena/agent,rating_q975,MISSING_COLUMN,double,,FILLED_WITH_NULL


occurred_at,component,severity,source,file_or_param,error_type,error_message
2026-10-10T09:01:56.391Z,bronze_notebook,ERROR,arena,source_dir=/Volumes/workspace/landing/staging/frontier/processed,RuntimeError,3 file(s) failed. See workspace.ops.error_logs. Re-run to retry only those.
2026-10-10T09:01:53.145Z,bronze_notebook,ERROR,arena,arena/agent__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T09:01:38.705Z,bronze_notebook,ERROR,arena,arena/text_style_control__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T09:01:36.596Z,bronze_notebook,ERROR,arena,arena/text__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T09:01:12.636Z,bronze_notebook,ERROR,all,source_dir=/Volumes/workspace/landing/staging/frontier/processed,RuntimeError,3 file(s) failed. See workspace.ops.error_logs. Re-run to retry only those.
2026-10-10T09:00:02.256Z,bronze_notebook,ERROR,arena,arena/agent__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T08:59:47.885Z,bronze_notebook,ERROR,arena,arena/text_style_control__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T08:59:45.700Z,bronze_notebook,ERROR,arena,arena/text__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found
2026-10-10T08:59:14.664Z,bronze_notebook,ERROR,all,source_dir=/Volumes/workspace/landing/staging/frontier/processed,RuntimeError,3 file(s) failed. See workspace.ops.error_logs. Re-run to retry only those.
2026-10-10T08:58:03.643Z,bronze_notebook,ERROR,arena,arena/agent__full__20261010T081407Z.parquet,FileMissing,File listed in the manifest was not found


### CELL 13 (optional) - Re-run quarantined files after you fixed the cause

In [0]:
# CELL 13 (optional) - Re-run quarantined files after you fixed the cause

RELEASE_FILES = []
for rel in RELEASE_FILES:
    source_batch = None
    for f in dbutils.fs.ls(f"{ROOT}/processed/_manifests"):
        with open(local(f.path), encoding="utf-8") as handle:
            m = json.load(handle)
        if any(x["path"] == rel for x in m["files"]):
            source_batch = m
    if source_batch is None:
        print("No manifest found for", rel)
        continue
    entry = [x for x in source_batch["files"] if x["path"] == rel][0]
    move(f"{ROOT}/quarantine/{rel}", f"{ROOT}/new/{rel}")
    release = dict(source_batch, batch_id=source_batch["batch_id"] + "_release", files=[entry])
    with open(f"{ROOT}/new/_manifests/manifest__{release['batch_id']}.json", "w", encoding="utf-8") as handle:
        json.dump(release, handle, indent=2)
    print("Released:", rel, "- run the notebook again to load it.")

In [0]:
display(spark.sql("""SELECT 'arena' t, COUNT(*) n FROM workspace.bronze.arena_leaderboard_raw
 UNION ALL SELECT 'cooperbench', COUNT(*) FROM workspace.bronze.cooperbench_member_raw
 UNION ALL SELECT 'benchlm', COUNT(*) FROM workspace.bronze.benchlm_raw
 UNION ALL SELECT 'epoch', COUNT(*) FROM workspace.bronze.epoch_models_raw"""))

t,n
arena,2200800
cooperbench,17328
benchlm,2383
epoch,3626


In [0]:
display(spark.sql("""SELECT layer, file_or_param, status, rows_read, rows_inserted, rows_updated,
 rows_skipped_duplicate, start_time, end_time FROM workspace.ops.pipeline_execution_logs
 ORDER BY load_timestamp DESC LIMIT 30"""))

layer,file_or_param,status,rows_read,rows_inserted,rows_updated,rows_skipped_duplicate,start_time,end_time
Raw-to-Bronze (run),source_dir=/Volumes/workspace/landing/staging/frontier/new,SUCCESS,1480,1480,0,0,2026-10-10T11:43:50.247Z,2026-10-10T11:45:09.371Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,3626,0,0,0,2026-10-10T11:45:06.092Z,2026-10-10T11:45:07.783Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,2383,0,0,0,2026-10-10T11:45:02.806Z,2026-10-10T11:45:04.526Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,17328,0,0,0,2026-10-10T11:44:58.682Z,2026-10-10T11:45:00.551Z
Bronze data quality (primary key),record_hash unique and not null,SUCCESS,2200800,0,0,0,2026-10-10T11:44:54.975Z,2026-10-10T11:44:57.002Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release_text.json,SUCCESS,0,0,0,0,2026-10-10T11:44:51.119Z,2026-10-10T11:44:51.729Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release.json,SUCCESS,0,0,0,0,2026-10-10T11:43:52.178Z,2026-10-10T11:44:47.671Z
Raw-to-Bronze,arena/agent__full__20261010T081407Z.parquet,SUCCESS,1480,1480,0,0,2026-10-10T11:43:52.531Z,2026-10-10T11:44:44.756Z
Raw-to-Bronze (run),source_dir=/Volumes/workspace/landing/staging/frontier/new,SUCCESS,0,0,0,0,2026-10-10T11:33:54.488Z,2026-10-10T11:34:20.051Z
Raw-to-Bronze (batch),/Volumes/workspace/landing/staging/frontier/new/_manifests/manifest__20261010T081407Z_full_release_text.json,SUCCESS,0,0,0,0,2026-10-10T11:34:04.819Z,2026-10-10T11:34:06.163Z


In [0]:
display(spark.sql("""SELECT DISTINCT batch_id, start_time, status FROM workspace.ops.pipeline_execution_logs
 WHERE layer = 'Raw-to-Bronze (batch)' ORDER BY start_time"""))

batch_id,start_time,status
20261010T081407Z_full,2026-10-10T08:20:14.493Z,REJECTED
20261010T081407Z_full,2026-10-10T08:29:50.659Z,SUCCESS
20261010T081407Z_full,2026-10-10T08:55:12.440Z,FAILURE
20261010T081407Z_full,2026-10-10T08:57:45.880Z,FAILURE
20261010T081407Z_full,2026-10-10T08:59:45.344Z,FAILURE
20261010T081407Z_full,2026-10-10T09:01:36.290Z,FAILURE
20261010T081407Z_full_release,2026-10-10T09:18:35.056Z,SUCCESS
20261010T081407Z_full_release_text,2026-10-10T09:27:45.657Z,SUCCESS
20261010T081407Z_full_release_text,2026-10-10T11:34:04.819Z,SUCCESS
20261010T081407Z_full_release,2026-10-10T11:43:52.178Z,SUCCESS


In [0]:
%sql
SELECT subset, COUNT(*) AS rows_loaded
FROM workspace.bronze.arena_leaderboard_raw
GROUP BY subset
ORDER BY subset;

subset,rows_loaded
agent,1480
text,1157599
text_style_control,1016009
webdev,25712


In [0]:
%sql
SELECT * FROM workspace.bronze.arena_leaderboard_raw LIMIT 20;

subset,load_type,model_name,organization,license,rating,variance,rating_q975,rating_q025,vote_count,rank,category,leaderboard_publish_date,raw_payload,record_hash,source_file,batch_id,collection_timestamp,load_timestamp,rating_lower,rating_upper,score,score_ci_lower,score_ci_upper,observation_count,session_count
text,full,llama2-70b-steerlm-chat,nvidia,Llama 2 Community,1092.6443627931317,564.1173450924275,null,null,195,29,chinese,2024-03-27,"{""category"":""chinese"",""leaderboard_publish_date"":""2024-03-27"",""license"":""Llama 2 Community"",""model_name"":""llama2-70b-steerlm-chat"",""organization"":""nvidia"",""rank"":29.0,""rating"":1092.6443627931317,""rating_lower"":1049.9832323166565,""rating_upper"":1141.941112393924,""variance"":564.1173450924275,""vote_count"":195.0}",ae893f3cb7956edff08de57622a7198a7c06aef21f0110f69a2b516ff351ce38,arena/text__full__20261010T081407Z.parquet,20261010T081407Z_full_release_text,2026-10-10T08:14:16.000Z,2026-10-10T09:27:52.954Z,1049.9832323166565,1141.941112393924,null,null,null,null,null
text,full,qwen-14b-chat,alibaba,Qianwen LICENSE,1135.1523779145336,477.4536863142253,null,null,230,29,chinese,2024-04-19,"{""category"":""chinese"",""leaderboard_publish_date"":""2024-04-19"",""license"":""Qianwen LICENSE"",""model_name"":""qwen-14b-chat"",""organization"":""alibaba"",""rank"":29.0,""rating"":1135.1523779145336,""rating_lower"":1094.00881950818,""rating_upper"":1172.1567647126299,""variance"":477.4536863142253,""vote_count"":230.0}",a158a07edfb4189ebab1fdea63af05911df6b221f373c8ac6d1a58bc76827051,arena/text__full__20261010T081407Z.parquet,20261010T081407Z_full_release_text,2026-10-10T08:14:16.000Z,2026-10-10T09:27:52.954Z,1094.00881950818,1172.1567647126299,null,null,null,null,null
text,full,llama-2-7b-chat,meta,Llama 2 Community,1034.6206307454272,97.96782350494769,null,null,1123,65,chinese,2024-05-01,"{""category"":""chinese"",""leaderboard_publish_date"":""2024-05-01"",""license"":""Llama 2 Community"",""model_name"":""llama-2-7b-chat"",""organization"":""meta"",""rank"":65.0,""rating"":1034.6206307454272,""rating_lower"":1013.2615144300723,""rating_upper"":1051.2764922546598,""variance"":97.96782350494769,""vote_count"":1123.0}",877e8d300b34f0ac6e6c7212d5049a2e14ffb512efeedafdb1a1d980c3d2a736,arena/text__full__20261010T081407Z.parquet,20261010T081407Z_full_release_text,2026-10-10T08:14:16.000Z,2026-10-10T09:27:52.954Z,1013.2615144300723,1051.2764922546598,null,null,null,null,null
text,full,mpt-30b-chat,MosaicML,CC-BY-NC-SA-4.0,1024.0613557629674,993.2731394423699,null,null,111,76,chinese,2024-06-06,"{""category"":""chinese"",""leaderboard_publish_date"":""2024-06-06"",""license"":""CC-BY-NC-SA-4.0"",""model_name"":""mpt-30b-chat"",""organization"":""MosaicML"",""rank"":76.0,""rating"":1024.0613557629674,""rating_lower"":963.3402298466943,""rating_upper"":1081.2328371145677,""variance"":993.2731394423699,""vote_count"":111.0}",4575bc3ab2f11ef0b7940b0445b6fc4b40bd8331e08ac5502abefdce67802d06,arena/text__full__20261010T081407Z.parquet,20261010T081407Z_full_release_text,2026-10-10T08:14:16.000Z,2026-10-10T09:27:52.954Z,963.3402298466943,1081.2328371145677,null,null,null,null,null
text,full,gemma-1.1-2b-it,google,Gemma license,1075.8920448843428,86.1276452420049,null,null,1336,60,chinese,2024-06-11,"{""category"":""chinese"",""leaderboard_publish_date"":""2024-06-11"",""license"":""Gemma license"",""model_name"":""gemma-1.1-2b-it"",""organization"":""google"",""rank"":60.0,""rating"":1075.8920448843428,""rating_lower"":1058.2731688877618,""rating_upper"":1091.5420989709964,""variance"":86.1276452420049,""vote_count"":1336.0}",aa937818b6bae4a5a7eb12d6599ec9779889486831a634888abfdebcc0c8fc3d,arena/text__full__20261010T081407Z.parquet,20261010T081407Z_full_release_text,2026-10-10T08:14:16.000Z,2026-10-10T09:27:52.954Z,1058.2731688877618,1091.5420989709964,null,null,null,null,null
text,full,llama-2-13b-chat,meta,Llama 2 Community,1055.3316963991447,108.1777897157405,null,null,1228,69,chine